### Setup

In [1]:
question_type = "multiple-choice" # Options: "multiple-choice"
dataset = "ptbr" # Options: "ptpt", "ptbr"
prompt_language = dataset # Options: "ptpt", "ptbr", "en"
models = {"llama-3.3-70b-instruct", "qwen3-8b", "qwen3-14b", "qwen3-32b", "qwen3-30b-a3b", "qwen3-235b-a22b", "gemma-3-27b-it"}# "claude-haiku-4.5", "deepseek-chat-v3.1", "gemini-2.5-flash","gpt-5",

### Load Prompts

In [2]:
import json

prompts = []

prompts_file = f'{dataset}-{question_type}-prompts-prompt-language-{prompt_language}.json'
with open(f'prompts/{prompts_file}', 'r', encoding='utf-8') as f:
    prompts = json.load(f)

### Load Model Answers

In [3]:
import glob
import os

base_runs_dir = f"results/{question_type}"
run_dirs = sorted(
    d for d in glob.glob(f"{base_runs_dir}/*") if os.path.isdir(d)
)
print("Detected runs:", run_dirs)
responses = {}

Detected runs: ['results/multiple-choice/first_run', 'results/multiple-choice/second_run']


In [4]:
for run_dir in run_dirs:
    run_name = os.path.basename(run_dir)
    responses[run_name] = {model: [] for model in models}

    for model in models:
        responses_file = f"{model}-{dataset}-responses-prompt-language-{prompt_language}.json"
        file_path = f"{run_dir}/{responses_file}"

        if not os.path.exists(file_path):
            continue

        with open(file_path, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    responses[run_name][model].append(json.loads(line))

print("Loaded responses for runs:", list(responses.keys()))


Loaded responses for runs: ['first_run', 'second_run']


### Parse Final Answers

In [5]:
import re

for run_name in responses:
    for model in models:
        for response in responses[run_name][model]:
            try:
                text = response["raw_response"]["choice.message.content"]
            except KeyError:
                response["final_answer"] = None
                continue

            m = re.search(r'(?<=\\boxed\{)([A-Za-z])(?=\}(?!.*\\boxed))', text)
            if m:
                response["final_answer"] = m.group(0)
            else:
                response["final_answer"] = None

### Calculate accuracies

In [6]:
from collections import defaultdict
import statistics

# Build golden lookup by question id
prompts_by_id = {item["id"]: item for item in prompts}

def init_stats():
    return {model: {lvl: {"correct": 0, "total": 0} for lvl in range(1, 5)} for model in models}

def compute_stats_for_run(responses_for_run):
    stats_with_fig = init_stats()
    stats_no_fig = init_stats()
    stats_all = init_stats()

    for model in models:
        for r in responses_for_run.get(model, []):
            qid = r.get("id")
            p = prompts_by_id.get(qid)
            if p is None:
                continue

            try:
                lvl = int(p.get("level", 0))
            except:
                continue
            if lvl not in (1, 2, 3, 4):
                continue

            contains_figure = bool(p.get("contains_latex_figure_in_question"))

            final = r.get("final_answer")
            correct_opt = p.get("correct_answer")
            is_correct = (
                final is not None
                and correct_opt is not None
                and str(final).strip().upper() == str(correct_opt).strip().upper()
            )

            stats_all[model][lvl]["total"] += 1
            if is_correct:
                stats_all[model][lvl]["correct"] += 1

            stats = stats_with_fig if contains_figure else stats_no_fig
            stats[model][lvl]["total"] += 1
            if is_correct:
                stats[model][lvl]["correct"] += 1

    return (
        compute_acc(stats_all),
        compute_acc(stats_with_fig),
        compute_acc(stats_no_fig),
    )


def compute_acc(stats):
    out = {}
    for model in models:
        out[model] = {}
        for lvl in range(1, 5):
            c = stats[model][lvl]["correct"]
            t = stats[model][lvl]["total"]
            out[model][lvl] = {
                "correct": c,
                "total": t,
                "accuracy": (c / t * 100.0) if t else None,
            }
    return out

def aggregate_runs(run_dict):
    out = {model: {lvl: {} for lvl in range(1, 5)} for model in models}

    for model in models:
        for lvl in range(1, 5):
            values = []
            for run_name in run_dict:
                acc = run_dict[run_name][model][lvl]["accuracy"]
                if acc is not None:
                    values.append(acc)

            if values:
                out[model][lvl]["mean"] = statistics.mean(values)
                out[model][lvl]["std"] = statistics.pstdev(values)
                out[model][lvl]["runs"] = values
            else:
                out[model][lvl]["mean"] = None
                out[model][lvl]["std"] = None
                out[model][lvl]["runs"] = []
    return out

def format_agg(title, agg):
    lines = [f"=== {title} ==="]
    for model in sorted(models):
        lines.append(f"\nModel: {model}")
        for lvl in range(1, 5):
            stats = agg[model][lvl]
            if stats["mean"] is None:
                lines.append(f"  Level {lvl}: no data")
            else:
                lines.append(
                    f"  Level {lvl}: mean={stats['mean']:.2f}%, "
                    f"std={stats['std']:.2f}%, "
                    f"runs={stats['runs']}"
                )
    lines.append("")
    return "\n".join(lines)


In [7]:
all_runs_acc_all = {}
all_runs_acc_with = {}
all_runs_acc_without = {}

for run_name in responses:
    acc_all, acc_with, acc_without = compute_stats_for_run(responses[run_name])
    all_runs_acc_all[run_name] = acc_all
    all_runs_acc_with[run_name] = acc_with
    all_runs_acc_without[run_name] = acc_without

agg_all = aggregate_runs(all_runs_acc_all)
agg_with = aggregate_runs(all_runs_acc_with)
agg_without = aggregate_runs(all_runs_acc_without)

report_text = "\n".join([
    format_agg("Overall (fig and no fig)", agg_all),
    format_agg("With figures only", agg_with),
    format_agg("Without figures", agg_without),
])

out_filename = (
    f"{dataset}-{question_type}-all-models-"
    f"prompt-language-{prompt_language}-acc-report-with-variability.txt"
)
out_path = f"results/accuracy-reports/{out_filename}"

os.makedirs("results/accuracy-reports", exist_ok=True)
with open(out_path, "w", encoding="utf-8") as out:
    out.write(report_text)

print("Saved variability report to:", out_path)

Saved variability report to: results/accuracy-reports/ptbr-multiple-choice-all-models-prompt-language-ptbr-acc-report-with-variability.txt
